# PII Metadata Pipeline — Tagging + Anonymization + De-anonymization
**Stack:** ydata-profiling + Microsoft Presidio + Fernet Encryption + SQL Server

### Tables produced
| Table | Contents |
|---|---|
| `metadata_dataset_overview` | Row/col counts, duplicates, missing % |
| `metadata_column_stats` | Type, missing count/%, unique count/% |
| `metadata_descriptive_stats` | mean, std, min, max, quartiles, skewness, kurtosis |
| `metadata_correlations` | Pairwise Pearson correlations |
| `metadata_pii_column_summary` | Per-column PII flags, entity types, confidence scores |
| `metadata_pii_cell_detail` | Per-cell detections with entity type, confidence, masked + encrypted values |
| `metadata_pii_anonymized` | Full DataFrame with PII columns masked |
| `metadata_pii_encryption_vault` | Encryption key registry for de-anonymization |

### Anonymization strategy
- **Mask** — visible safe version stored in `metadata_pii_anonymized` and shown in reports
- **Encrypt** — reversible Fernet encryption stored in `metadata_pii_cell_detail` and vault
- **De-anonymize** — decrypt back to original using stored key from vault

## Cell 1 — Install dependencies
Run once then **restart the kernel**.

In [ ]:
import sys
!{sys.executable} -m pip install -q ydata-profiling presidio-analyzer presidio-anonymizer pyodbc sqlalchemy faker cryptography
!{sys.executable} -m spacy download en_core_web_lg
print("\n✅ Installation complete — RESTART THE KERNEL now, then run from Cell 2.")

## Cell 2 — Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import random
import base64
from datetime import datetime
from collections import defaultdict

# ydata-profiling
from ydata_profiling import ProfileReport

# Presidio — detection
from presidio_analyzer import AnalyzerEngine, RecognizerRegistry
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_analyzer.predefined_recognizers import (
    EmailRecognizer, PhoneRecognizer, CreditCardRecognizer,
    IpRecognizer, UsSsnRecognizer, DateRecognizer, SpacyRecognizer,
)

# Presidio — anonymization
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import RecognizerResult, OperatorConfig

# Fernet encryption (reversible)
from cryptography.fernet import Fernet

# Faker for synthetic data
from faker import Faker

print("✅ Imports OK")

## Cell 3 — Verify spaCy model

In [ ]:
import spacy
installed = spacy.util.get_installed_models()
print("Installed spaCy models:", installed)

if "en_core_web_lg" in installed:
    SPACY_MODEL = "en_core_web_lg"
elif "en_core_web_md" in installed:
    SPACY_MODEL = "en_core_web_md"
elif "en_core_web_sm" in installed:
    SPACY_MODEL = "en_core_web_sm"
else:
    raise EnvironmentError("No spaCy English model found. Run Cell 1 and restart the kernel.")

print(f"✅ Using spaCy model: {SPACY_MODEL}")

## Cell 4 — Build Presidio analyzer + anonymizer + smoke test

In [ ]:
# ── Analyzer ──────────────────────────────────────────────────────────────────
nlp_config = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "en", "model_name": SPACY_MODEL}],
}
nlp_engine = NlpEngineProvider(nlp_configuration=nlp_config).create_engine()

registry = RecognizerRegistry()
registry.add_recognizer(EmailRecognizer(supported_language="en"))
registry.add_recognizer(PhoneRecognizer(supported_language="en", supported_regions=["US", "GB", "CA"]))
registry.add_recognizer(CreditCardRecognizer(supported_language="en"))
registry.add_recognizer(IpRecognizer(supported_language="en"))
registry.add_recognizer(UsSsnRecognizer(supported_language="en"))
registry.add_recognizer(DateRecognizer(supported_language="en"))
registry.add_recognizer(SpacyRecognizer(
    supported_language="en",
    supported_entities=["PERSON", "LOCATION"],
    ner_strength=0.85,
))

analyzer  = AnalyzerEngine(nlp_engine=nlp_engine, registry=registry, supported_languages=["en"])
anonymizer = AnonymizerEngine()

print("Registered recognizers:")
for r in analyzer.registry.recognizers:
    print(f"  {r.__class__.__name__:40s} → {r.supported_entities}")

# ── Smoke test ────────────────────────────────────────────────────────────────
smoke_tests = [
    ("EMAIL",       "alice@example.com"),
    ("SSN",         "123-45-6789"),
    ("PHONE",       "+1-800-555-0199"),
    ("CREDIT CARD", "4111111111111111"),
    ("IP",          "192.168.1.1"),
    ("DATE",        "1990-04-15"),
]
print("\nSmoke tests:")
all_passed = True
for label, text in smoke_tests:
    results = analyzer.analyze(text=text, language="en")
    if results:
        best = max(results, key=lambda r: r.score)
        print(f"  ✅ [{label:12s}] '{text}' → {best.entity_type} (score={best.score:.2f})")
    else:
        print(f"  ❌ [{label:12s}] '{text}' → NO RESULT")
        all_passed = False
print("\n✅ All smoke tests passed!" if all_passed else "\n⚠️  Some tests failed.")

## Cell 5 — Encryption key setup
A single Fernet key is generated per run and stored in the vault table.
**Keep this key safe — it is required for de-anonymization.**

In [ ]:
# ── Generate encryption key ───────────────────────────────────────────────────
FERNET_KEY = Fernet.generate_key()          # bytes
fernet     = Fernet(FERNET_KEY)
KEY_ID     = f"key_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"

print(f"Key ID     : {KEY_ID}")
print(f"Fernet Key : {FERNET_KEY.decode()}")
print("\n⚠️  Save the key above — you need it to de-anonymize outside this session.")

def encrypt_value(val: str) -> str:
    """Encrypt a string value — returns base64 token string."""
    return fernet.encrypt(val.encode()).decode()

def decrypt_value(token: str) -> str:
    """Decrypt a Fernet token back to original string."""
    return fernet.decrypt(token.encode()).decode()

def mask_value(val: str, entity_type: str) -> str:
    """
    Mask a PII value based on its entity type.
    Preserves structure where useful (e.g. email domain, last 4 of CC).
    """
    val = str(val)
    if entity_type == "EMAIL_ADDRESS" and "@" in val:
        local, domain = val.split("@", 1)
        return f"{'*' * len(local)}@{domain}"
    elif entity_type == "CREDIT_CARD":
        digits = val.replace("-", "").replace(" ", "")
        return f"{'*' * (len(digits) - 4)}{digits[-4:]}"
    elif entity_type == "US_SSN":
        return "***-**-" + val[-4:] if len(val) >= 4 else "***-**-****"
    elif entity_type == "PHONE_NUMBER":
        digits = [c for c in val if c.isdigit()]
        return "*" * (len(digits) - 4) + "".join(digits[-4:]) if len(digits) >= 4 else "*" * len(val)
    elif entity_type == "IP_ADDRESS":
        parts = val.split(".")
        return "*.*.*" + "." + parts[-1] if len(parts) == 4 else "*.*.*.*"
    else:
        # Generic: keep first and last char, mask the middle
        if len(val) <= 2:
            return "*" * len(val)
        return val[0] + "*" * (len(val) - 2) + val[-1]

print("\n✅ Encryption helpers ready")

## Cell 6 — Generate synthetic data

In [ ]:
fake = Faker("en_US")
Faker.seed(42)
random.seed(42)
np.random.seed(42)

N = 50

def maybe_none(val, pct_missing=0.15):
    return None if random.random() < pct_missing else val

records = []
for _ in range(N):
    records.append({
        "full_name":   maybe_none(fake.name()),
        "email":       maybe_none(fake.email()),
        "phone":       maybe_none(fake.phone_number()),
        "ssn":         maybe_none(fake.ssn()),
        "credit_card": maybe_none(fake.credit_card_number()),
        "dob":         maybe_none(str(fake.date_of_birth(minimum_age=18, maximum_age=70))),
        "ip_address":  maybe_none(fake.ipv4_private()),
        "address":     maybe_none(fake.address().replace("\n", ", ")),
        "department":  maybe_none(random.choice(["HR", "Engineering", "Finance", "Legal", "Marketing"])),
        "salary":      maybe_none(round(np.random.normal(75000, 15000), 2)),
        "tenure_years":maybe_none(round(random.uniform(0.5, 20.0), 1)),
        "performance": maybe_none(round(np.random.uniform(1.0, 5.0), 1)),
    })

df           = pd.DataFrame(records)
DATASET_NAME = "synthetic_employees"
RUN_TS       = datetime.utcnow()

print(f"✅ Synthetic dataset: {df.shape[0]} rows × {df.shape[1]} cols")
df.head(3)

## Cell 7 — Run ydata-profiling

In [ ]:
profile = ProfileReport(df, title=DATASET_NAME, explorative=True, minimal=False)
# profile.to_notebook_iframe()  # uncomment to view full report
print("✅ Profiling complete")

## Cell 8 — Extract ydata-profiling metadata

In [ ]:
desc        = profile.get_description()
table_stats = desc.table
n_rows      = int(table_stats.get("n", 1))

overview_df = pd.DataFrame([{
    "dataset_name":       DATASET_NAME, "run_timestamp": RUN_TS,
    "n_rows":             n_rows,
    "n_cols":             int(table_stats.get("n_var", 0)),
    "n_missing_cells":    int(table_stats.get("n_cells_missing", 0)),
    "pct_missing_cells":  round(float(table_stats.get("p_cells_missing", 0)) * 100, 4),
    "n_duplicate_rows":   int(table_stats.get("n_duplicates", 0)),
    "pct_duplicate_rows": round(float(table_stats.get("p_duplicates", 0)) * 100, 4),
    "n_numeric_cols":     int(table_stats.get("types", {}).get("Numeric", 0)),
    "n_categorical_cols": int(table_stats.get("types", {}).get("Categorical", 0)),
}])

col_stats_rows, desc_stats_rows = [], []
for col_name, col_data in desc.variables.items():
    col_type = str(col_data.get("type", "Unknown"))
    col_stats_rows.append({
        "dataset_name": DATASET_NAME, "run_timestamp": RUN_TS,
        "column_name": col_name, "column_type": col_type,
        "n_missing":   int(col_data.get("n_missing", 0)),
        "pct_missing": round(float(col_data.get("p_missing", 0)) * 100, 4),
        "n_unique":    int(col_data.get("n_unique", col_data.get("n_distinct", 0))),
        "pct_unique":  round(int(col_data.get("n_unique", col_data.get("n_distinct", 0))) / n_rows * 100, 4),
    })
    if col_type in ("Numeric", "NUM") or col_data.get("mean") is not None:
        desc_stats_rows.append({
            "dataset_name": DATASET_NAME, "run_timestamp": RUN_TS,
            "column_name": col_name,
            "mean":     round(float(col_data.get("mean",     0) or 0), 6),
            "std":      round(float(col_data.get("std",      0) or 0), 6),
            "min":      round(float(col_data.get("min",      0) or 0), 6),
            "pct_25":   round(float(col_data.get("25%",      0) or 0), 6),
            "median":   round(float(col_data.get("50%",      0) or 0), 6),
            "pct_75":   round(float(col_data.get("75%",      0) or 0), 6),
            "max":      round(float(col_data.get("max",      0) or 0), 6),
            "kurtosis": round(float(col_data.get("kurtosis", 0) or 0), 6),
            "skewness": round(float(col_data.get("skewness", 0) or 0), 6),
        })

col_stats_df  = pd.DataFrame(col_stats_rows)
desc_stats_df = pd.DataFrame(desc_stats_rows)

corr_rows = []
if desc.correlations:
    corr_key = "pearson" if "pearson" in desc.correlations else next(iter(desc.correlations), None)
    if corr_key:
        corr_matrix = desc.correlations[corr_key]
        for col_a in corr_matrix.columns:
            for col_b in corr_matrix.columns:
                if col_a < col_b:
                    val = corr_matrix.loc[col_a, col_b] if col_a in corr_matrix.index else None
                    if val is not None and not pd.isna(val):
                        corr_rows.append({
                            "dataset_name": DATASET_NAME, "run_timestamp": RUN_TS,
                            "correlation_type": corr_key,
                            "column_a": col_a, "column_b": col_b,
                            "correlation_value": round(float(val), 6),
                        })
corr_df = pd.DataFrame(corr_rows)

print(f"✅ Profiling metadata extracted — overview: {len(overview_df)} | "
      f"col_stats: {len(col_stats_df)} | desc_stats: {len(desc_stats_df)} | correlations: {len(corr_df)}")

## Cell 9 — PII detection + tagging + masking + encryption

In [ ]:
ENTITY_THRESHOLDS = {
    "EMAIL_ADDRESS": 0.5, "US_SSN":       0.5,
    "PHONE_NUMBER":  0.5, "CREDIT_CARD":  0.5,
    "IP_ADDRESS":    0.5, "DATE_TIME":    0.6,
    "PERSON":        0.7, "LOCATION":     0.7,
}
PII_ENTITIES   = list(ENTITY_THRESHOLDS.keys())
string_cols    = df.select_dtypes(include=["object", "string"]).columns.tolist()

# ── Output structures ─────────────────────────────────────────────────────────
pii_cell_rows   = []          # per-cell detail with mask + encrypted value
pii_col_staging = defaultdict(list)

# Anonymized DataFrame — start with a copy, replace PII cells with masked value
df_masked = df.copy()

# Track which (row, col) cells have been masked to avoid double-masking
masked_cells = set()

print(f"Scanning {len(string_cols)} string column(s)...\n")

for col in string_cols:
    col_detections = 0
    for row_idx, cell_value in df[col].items():
        if pd.isna(cell_value) or str(cell_value).strip() == "":
            continue

        try:
            results = analyzer.analyze(
                text=str(cell_value), entities=PII_ENTITIES, language="en"
            )
        except Exception as e:
            print(f"  ⚠️  Error on col='{col}' row={row_idx}: {e}")
            continue

        for result in results:
            threshold = ENTITY_THRESHOLDS.get(result.entity_type, 0.5)
            if result.score < threshold:
                continue

            original_val  = str(cell_value)
            masked_val    = mask_value(original_val, result.entity_type)
            encrypted_val = encrypt_value(original_val)

            pii_cell_rows.append({
                "dataset_name":    DATASET_NAME,
                "run_timestamp":   RUN_TS,
                "column_name":     col,
                "row_index":       int(row_idx),
                "original_value":  original_val,
                "masked_value":    masked_val,
                "encrypted_value": encrypted_val,
                "entity_type":     result.entity_type,
                "confidence":      round(result.score, 6),
                "start_char":      result.start,
                "end_char":        result.end,
                "key_id":          KEY_ID,
            })

            pii_col_staging[col].append((result.entity_type, result.score))
            col_detections += 1

            # Apply mask to anonymized DataFrame (first detection per cell wins)
            if (row_idx, col) not in masked_cells:
                df_masked.at[row_idx, col] = masked_val
                masked_cells.add((row_idx, col))

    print(f"  {col:20s} → {col_detections} detection(s)")

pii_cell_df = pd.DataFrame(pii_cell_rows) if pii_cell_rows else pd.DataFrame(
    columns=["dataset_name","run_timestamp","column_name","row_index",
             "original_value","masked_value","encrypted_value",
             "entity_type","confidence","start_char","end_char","key_id"]
)

print(f"\n✅ Total PII detections: {len(pii_cell_df)} | Cells masked: {len(masked_cells)}")

## Cell 10 — Build per-column PII summary

In [ ]:
pii_col_rows = []
for col in string_cols:
    detections   = pii_col_staging.get(col, [])
    n_non_null   = int(df[col].notna().sum())
    entity_types = sorted({e for e, _ in detections})
    scores       = [s for _, s in detections]
    flagged_rows = (
        pii_cell_df[pii_cell_df["column_name"] == col]["row_index"].nunique()
        if not pii_cell_df.empty else 0
    )
    pii_col_rows.append({
        "dataset_name":          DATASET_NAME,
        "run_timestamp":         RUN_TS,
        "column_name":           col,
        "is_pii":                len(entity_types) > 0,
        "detected_entity_types": ", ".join(entity_types) if entity_types else None,
        "n_entity_types":        len(entity_types),
        "avg_confidence":        round(float(np.mean(scores)), 6) if scores else None,
        "max_confidence":        round(float(np.max(scores)),  6) if scores else None,
        "min_confidence":        round(float(np.min(scores)),  6) if scores else None,
        "n_flagged_rows":        flagged_rows,
        "pct_flagged_rows":      round(flagged_rows / n_non_null * 100, 4) if n_non_null > 0 else 0.0,
        "anonymization_method":  "mask + encrypt",
        "thresholds_applied":    str(ENTITY_THRESHOLDS),
    })

pii_col_df = pd.DataFrame(pii_col_rows)
print("✅ Per-column PII summary:")
display(pii_col_df[[
    "column_name","is_pii","detected_entity_types",
    "n_flagged_rows","pct_flagged_rows","avg_confidence"
]])

## Cell 11 — Build anonymized DataFrame + encryption vault

In [ ]:
# ── Anonymized DataFrame (masked values) ──────────────────────────────────────
df_anonymized = df_masked.copy()
df_anonymized.insert(0, "dataset_name",  DATASET_NAME)
df_anonymized.insert(1, "run_timestamp", RUN_TS)
df_anonymized.insert(2, "original_row_index", df_anonymized.index)

print("✅ Anonymized DataFrame (first 5 rows):")
display(df_anonymized.head())

# ── Encryption vault ──────────────────────────────────────────────────────────
vault_df = pd.DataFrame([{
    "key_id":        KEY_ID,
    "dataset_name":  DATASET_NAME,
    "run_timestamp": RUN_TS,
    "fernet_key":    FERNET_KEY.decode(),   # store as string
    "entities_covered": ", ".join(PII_ENTITIES),
    "created_by":    "pii_metadata_pipeline",
    "note":          "Keep this key secure. Required for de-anonymization."
}])

print("\n✅ Encryption vault:")
display(vault_df[["key_id","dataset_name","run_timestamp","entities_covered"]])

## Cell 12 — De-anonymization demo

In [ ]:
def deanonymize_dataframe(df_anon: pd.DataFrame, pii_cell_df: pd.DataFrame, fernet_key: bytes) -> pd.DataFrame:
    """
    Restore original PII values from encrypted tokens stored in pii_cell_df.
    Uses the Fernet key to decrypt each token and writes back to the correct cell.
    """
    f          = Fernet(fernet_key)
    df_restored = df_anon.copy()
    restored    = 0

    for _, det_row in pii_cell_df.iterrows():
        col       = det_row["column_name"]
        row_idx   = det_row["row_index"]
        enc_token = det_row["encrypted_value"]

        if col not in df_restored.columns:
            continue
        try:
            original = f.decrypt(enc_token.encode()).decode()
            df_restored.at[row_idx, col] = original
            restored += 1
        except Exception as e:
            print(f"  ⚠️  Could not decrypt row={row_idx} col='{col}': {e}")

    print(f"✅ De-anonymized {restored} cell(s)")
    return df_restored


# ── Run de-anonymization using the key from this session ──────────────────────
# Drop metadata columns added in Cell 11 before restoring
df_to_restore = df_anonymized.drop(columns=["dataset_name","run_timestamp","original_row_index"])
df_restored   = deanonymize_dataframe(df_to_restore, pii_cell_df, FERNET_KEY)

print("\nOriginal vs Anonymized vs Restored (first 5 rows):")
compare_cols = [c for c in ["full_name","email","ssn","credit_card"] if c in df.columns]

comparison = pd.DataFrame()
for col in compare_cols:
    comparison[f"{col}_original"]    = df[col].head()
    comparison[f"{col}_masked"]      = df_to_restore[col].head()
    comparison[f"{col}_restored"]    = df_restored[col].head()

display(comparison)

# ── Validate round-trip ───────────────────────────────────────────────────────
mismatches = 0
for col in string_cols:
    if col not in df_restored.columns:
        continue
    for row_idx in df.index:
        orig = df.at[row_idx, col]
        rest = df_restored.at[row_idx, col]
        if pd.notna(orig) and orig != rest:
            mismatches += 1

if mismatches == 0:
    print("\n✅ Round-trip validation passed — all values restored correctly")
else:
    print(f"\n⚠️  {mismatches} value(s) did not restore correctly — check encryption vault")

## Cell 13 — SQL Server connection

In [ ]:
from sqlalchemy import create_engine, text
import urllib

SQL_SERVER   = "YOUR_SERVER_NAME"
SQL_DATABASE = "YOUR_DATABASE_NAME"
SQL_SCHEMA   = "dbo"

# Option A: Windows Authentication
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SQL_SERVER};DATABASE={SQL_DATABASE};Trusted_Connection=yes;"
)

# Option B: Username & Password
# SQL_USERNAME = "YOUR_USERNAME"
# SQL_PASSWORD = "YOUR_PASSWORD"
# params = urllib.parse.quote_plus(
#     f"DRIVER={{ODBC Driver 17 for SQL Server}};"
#     f"SERVER={SQL_SERVER};DATABASE={SQL_DATABASE};"
#     f"UID={SQL_USERNAME};PWD={SQL_PASSWORD};"
# )

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("✅ SQL Server connection successful")

## Cell 14 — Auto-create all SQL Server tables

In [ ]:
DDL_STATEMENTS = [
f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name='metadata_dataset_overview')
CREATE TABLE [{SQL_SCHEMA}].[metadata_dataset_overview] (
    id INT IDENTITY(1,1) PRIMARY KEY, dataset_name NVARCHAR(255) NOT NULL,
    run_timestamp DATETIME2 NOT NULL, n_rows INT, n_cols INT,
    n_missing_cells INT, pct_missing_cells FLOAT,
    n_duplicate_rows INT, pct_duplicate_rows FLOAT,
    n_numeric_cols INT, n_categorical_cols INT);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name='metadata_column_stats')
CREATE TABLE [{SQL_SCHEMA}].[metadata_column_stats] (
    id INT IDENTITY(1,1) PRIMARY KEY, dataset_name NVARCHAR(255) NOT NULL,
    run_timestamp DATETIME2 NOT NULL, column_name NVARCHAR(255) NOT NULL,
    column_type NVARCHAR(100), n_missing INT, pct_missing FLOAT,
    n_unique INT, pct_unique FLOAT);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name='metadata_descriptive_stats')
CREATE TABLE [{SQL_SCHEMA}].[metadata_descriptive_stats] (
    id INT IDENTITY(1,1) PRIMARY KEY, dataset_name NVARCHAR(255) NOT NULL,
    run_timestamp DATETIME2 NOT NULL, column_name NVARCHAR(255) NOT NULL,
    mean FLOAT, std FLOAT, min FLOAT, pct_25 FLOAT, median FLOAT,
    pct_75 FLOAT, max FLOAT, kurtosis FLOAT, skewness FLOAT);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name='metadata_correlations')
CREATE TABLE [{SQL_SCHEMA}].[metadata_correlations] (
    id INT IDENTITY(1,1) PRIMARY KEY, dataset_name NVARCHAR(255) NOT NULL,
    run_timestamp DATETIME2 NOT NULL, correlation_type NVARCHAR(50),
    column_a NVARCHAR(255) NOT NULL, column_b NVARCHAR(255) NOT NULL,
    correlation_value FLOAT);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name='metadata_pii_column_summary')
CREATE TABLE [{SQL_SCHEMA}].[metadata_pii_column_summary] (
    id INT IDENTITY(1,1) PRIMARY KEY, dataset_name NVARCHAR(255) NOT NULL,
    run_timestamp DATETIME2 NOT NULL, column_name NVARCHAR(255) NOT NULL,
    is_pii BIT NOT NULL, detected_entity_types NVARCHAR(500),
    n_entity_types INT, avg_confidence FLOAT, max_confidence FLOAT,
    min_confidence FLOAT, n_flagged_rows INT, pct_flagged_rows FLOAT,
    anonymization_method NVARCHAR(100), thresholds_applied NVARCHAR(MAX));""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name='metadata_pii_cell_detail')
CREATE TABLE [{SQL_SCHEMA}].[metadata_pii_cell_detail] (
    id INT IDENTITY(1,1) PRIMARY KEY, dataset_name NVARCHAR(255) NOT NULL,
    run_timestamp DATETIME2 NOT NULL, column_name NVARCHAR(255) NOT NULL,
    row_index INT NOT NULL, original_value NVARCHAR(MAX),
    masked_value NVARCHAR(MAX), encrypted_value NVARCHAR(MAX),
    entity_type NVARCHAR(100) NOT NULL, confidence FLOAT NOT NULL,
    start_char INT, end_char INT, key_id NVARCHAR(100));""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name='metadata_pii_anonymized')
CREATE TABLE [{SQL_SCHEMA}].[metadata_pii_anonymized] (
    id INT IDENTITY(1,1) PRIMARY KEY, dataset_name NVARCHAR(255),
    run_timestamp DATETIME2, original_row_index INT,
    full_name NVARCHAR(500), email NVARCHAR(500), phone NVARCHAR(100),
    ssn NVARCHAR(50), credit_card NVARCHAR(50), dob NVARCHAR(50),
    ip_address NVARCHAR(50), address NVARCHAR(MAX), department NVARCHAR(100),
    salary FLOAT, tenure_years FLOAT, performance FLOAT);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name='metadata_pii_encryption_vault')
CREATE TABLE [{SQL_SCHEMA}].[metadata_pii_encryption_vault] (
    id INT IDENTITY(1,1) PRIMARY KEY, key_id NVARCHAR(100) NOT NULL,
    dataset_name NVARCHAR(255) NOT NULL, run_timestamp DATETIME2 NOT NULL,
    fernet_key NVARCHAR(MAX) NOT NULL, entities_covered NVARCHAR(500),
    created_by NVARCHAR(255), note NVARCHAR(MAX));"""
]

with engine.begin() as conn:
    for stmt in DDL_STATEMENTS:
        conn.execute(text(stmt))
print("✅ All 8 tables created (or already exist)")

## Cell 15 — Insert all metadata into SQL Server

In [ ]:
TABLE_MAP = {
    "metadata_dataset_overview":    overview_df,
    "metadata_column_stats":        col_stats_df,
    "metadata_descriptive_stats":   desc_stats_df,
    "metadata_correlations":        corr_df,
    "metadata_pii_column_summary":  pii_col_df,
    "metadata_pii_cell_detail":     pii_cell_df,
    "metadata_pii_anonymized":      df_anonymized,
    "metadata_pii_encryption_vault":vault_df,
}

with engine.begin() as conn:
    for table_name, df_to_insert in TABLE_MAP.items():
        if df_to_insert.empty:
            print(f"  ⚠️  Skipped {table_name} (empty)")
            continue
        df_to_insert.to_sql(
            name=table_name, con=conn, schema=SQL_SCHEMA,
            if_exists="append", index=False
        )
        print(f"  ✅ Inserted {len(df_to_insert):>4} row(s) → [{SQL_SCHEMA}].[{table_name}]")

print("\n✅ All metadata written to SQL Server")

## Cell 16 — De-anonymize from SQL Server using vault key
Use this to restore original values in a future session using the stored key.

In [ ]:
# ── Load anonymized data + key from SQL Server ────────────────────────────────
df_from_sql = pd.read_sql(
    f"SELECT * FROM [{SQL_SCHEMA}].[metadata_pii_anonymized] WHERE dataset_name = '{DATASET_NAME}'",
    engine
)

vault_from_sql = pd.read_sql(
    f"SELECT * FROM [{SQL_SCHEMA}].[metadata_pii_encryption_vault] WHERE dataset_name = '{DATASET_NAME}'",
    engine
)

pii_detail_from_sql = pd.read_sql(
    f"SELECT * FROM [{SQL_SCHEMA}].[metadata_pii_cell_detail] WHERE dataset_name = '{DATASET_NAME}'",
    engine
)

# ── Retrieve the key and decrypt ──────────────────────────────────────────────
retrieved_key = vault_from_sql.iloc[-1]["fernet_key"].encode()  # latest key

df_sql_restored = deanonymize_dataframe(
    df_from_sql.drop(columns=["id","dataset_name","run_timestamp","original_row_index"], errors="ignore"),
    pii_detail_from_sql,
    retrieved_key
)

print("\n✅ Restored from SQL Server (first 5 rows):")
display(df_sql_restored[compare_cols].head())

## Cell 17 — Useful SQL queries
```sql
-- View masked data (safe to share)
SELECT * FROM dbo.metadata_pii_anonymized
WHERE dataset_name = 'synthetic_employees';

-- View PII detections with masked + encrypted values
SELECT column_name, row_index, masked_value, entity_type, confidence
FROM dbo.metadata_pii_cell_detail
WHERE dataset_name = 'synthetic_employees'
ORDER BY confidence DESC;

-- Retrieve encryption key for de-anonymization
SELECT key_id, fernet_key, entities_covered
FROM dbo.metadata_pii_encryption_vault
WHERE dataset_name = 'synthetic_employees';

-- Full column inventory with PII + anonymization status
SELECT cs.column_name, cs.column_type, cs.pct_missing,
       pii.is_pii, pii.detected_entity_types,
       pii.avg_confidence, pii.anonymization_method
FROM dbo.metadata_column_stats cs
LEFT JOIN dbo.metadata_pii_column_summary pii
       ON cs.column_name = pii.column_name
      AND cs.dataset_name = pii.dataset_name
      AND cs.run_timestamp = pii.run_timestamp
WHERE cs.dataset_name = 'synthetic_employees';
```